# TEM/STEM nanoparticle polydispersity — Thermo Fisher `.emd` files

Dedicated pipeline for **Thermo Fisher Velox `.emd`** files. Pixel calibration and acquisition mode (bright-field vs dark-field/HAADF) are read directly from the file's own metadata via HyperSpy — you only need to point it at a file (or a folder of files).

**Pipeline**: load `.emd` + metadata → normalize to 8-bit → Otsu threshold (direction chosen automatically from bright/dark-field) → watershed split of touching particles → measure diameters → polydispersity statistics + histogram + overlay.

**Dependencies**: `hyperspy`, `opencv-python-headless`, `numpy`, `scipy`, `scikit-image`, `matplotlib`, `pandas` (see `requirements.txt`).

**If no particles are found**: the pipeline does not crash — it shows you the segmentation mask instead, so you can see why (wrong bright/dark-field guess, or filters too strict) and adjust the parameters in Configuration.

## Configuration

Set `mode` to choose what to analyze: **`"file"`** runs the *Single file* section on `file_path`, **`"folder"`** runs *Batch processing* on every `.emd` file in `folder_path`. Only the section matching `mode` does anything, so **Run All Cells** is safe; the other section just prints a "Skipped" message. The remaining parameters have sensible defaults -- only touch them if particle detection doesn't work well (see the notes on each parameter).

**Particle filters**: `min_diam_nm` is a *physical* size (so it does not depend on the magnification, unlike a threshold in pixels). `max_intensity_ratio` removes overlapping particles: in dark-field/HAADF images the signal adds up where particles overlap, so overlapped particles are brighter than isolated ones; any particle whose mean intensity exceeds this multiple of the median particle intensity is excluded (drawn in orange on the overlay, not counted in the statistics). Set it to `None` to disable.

**Magnification-independent settings**: `min_peak_distance_nm` and `smooth_nm` are physical lengths, converted to pixels with each file's own calibration (a fixed number of pixels works at one magnification and fails at another). `exclude_border` drops particles cut by the image edge (cyan on the overlay), because a truncated particle has a wrong diameter. `emd_signal_index` is normally left at `None`: a `.emd` can contain several images (e.g. small auxiliary ones next to the main HAADF image) and the largest one is used automatically.

`max_diameter_ratio` removes oversized objects (merged clusters, aggregates mistaken for one particle): any particle whose diameter is more than this multiple of the mean diameter found (after the other filters) is excluded and drawn in magenta. Set it to `None` to disable.

In [ ]:
mode = "folder"  # "file" -> analyze the single file in file_path | "folder" -> analyze every file in folder_path

file_path = "./Data/Velox/STEM HAADF 1532.emd"  # <-- path to a single .emd file (used by "Single file" section)
folder_path = "./Data/Velox"                     # <-- folder containing .emd files (used by "Batch processing" section)
outdir = "Result_Poly_emd_2"  # set to None to skip saving overlay.png/histogram.png to disk

min_diam_nm = 5          # minimum accepted particle diameter, in nm (physical size: independent of magnification; noise fragments are smaller)
max_intensity_ratio = 1.3 # dark-field only: exclude particles brighter than this multiple of the median particle intensity (overlaps); None = off
max_diameter_ratio = 2.5  # exclude particles whose diameter is more than this multiple of the mean diameter found (merged clusters, aggregates); None = off
min_circularity = 0.5     # minimum accepted circularity (1.0 = perfect circle)
min_peak_distance_nm = 5  # minimum distance between particle centers for the watershed, in nm (scaled to each file's magnification; lower for small/closely packed particles)
smooth_nm = 0.3           # edge smoothing before thresholding, in nm (removes noisy edges at very high magnification; never below ~1 px)
exclude_border = True     # drop particles touching the image border (truncated, so their diameter would be underestimated); drawn in cyan
emd_signal_index = None   # None = automatic (largest image in the file); set 0, 1, 2... only to force a specific signal
bright_particles_override = None  # None = auto-detect bright/dark-field from metadata; set True or False to force it if the guess is wrong
outdir = "Result_Poly_emd_2"  # set to None to skip saving overlay.png/histogram.png to disk

assert mode in ("file", "folder"), "mode must be 'file' or 'folder'"

In [ ]:
import os
import glob

import cv2
import numpy as np
import pandas as pd
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops
import matplotlib.pyplot as plt
import hyperspy.api as hs

%matplotlib inline

## 1. Loading `.emd` files and metadata

`load_emd_image()` reads the pixel calibration (nm/px) and detector name from the file's metadata, normalizes the (often 16-bit-range) raw detector counts to 8-bit, and guesses bright-field vs dark-field from the detector name (e.g. `HAADF`, `DF-S` → dark-field; `BF-S` → bright-field). Override with `bright_particles_override` in Configuration if it guesses wrong for your detector naming.

In [ ]:
UNIT_TO_NM = {
    "nm": 1.0,
    "um": 1000.0,
    "\u00b5m": 1000.0,
    "pm": 0.001,
    "a": 0.1,
    "\u00e5": 0.1,
    "angstrom": 0.1,
    "m": 1e9,
}


def normalize_to_uint8(data, low_percentile=1.0, high_percentile=99.0):
    """
    Rescales an arbitrary-range array (e.g. raw STEM detector counts) to
    0-255 uint8, clipping the extreme percentiles first so a few outlier
    pixels don't compress the contrast of the rest of the image.
    """
    lo, hi = np.percentile(data, [low_percentile, high_percentile])
    if hi <= lo:
        lo, hi = float(data.min()), float(data.max())
    clipped = np.clip(data, lo, hi)
    scaled = (clipped - lo) / (hi - lo + 1e-12) * 255.0
    return scaled.astype(np.uint8)


def classify_darkfield(detector_name):
    """
    Guesses bright-field vs dark-field from a Thermo Fisher detector name
    (e.g. 'HAADF', 'DF-S' -> dark-field; 'BF-S' -> bright-field). Defaults
    to bright-field (False) if the name is missing or unrecognized.
    """
    if not detector_name:
        return False
    return "DF" in str(detector_name).upper()


def load_emd_image(path, signal_index=None):
    """
    Loads a Thermo Fisher Velox .emd file with HyperSpy and returns
    (gray_uint8, nm_per_px, is_darkfield, info). Calibration and
    acquisition mode are read from the file's metadata -- no scale bar
    detection needed. A .emd can contain several signals (e.g. small
    auxiliary images next to the main HAADF image): with signal_index=None
    the 2D signal with the most pixels is used; pass an integer to force one.
    """
    result = hs.load(path)
    signals = result if isinstance(result, list) else [result]
    if signal_index is None:
        sizes = [s_.data.size if s_.axes_manager.signal_dimension == 2 else 0 for s_ in signals]
        signal_index = int(np.argmax(sizes))
    if len(signals) > 1:
        print("Note: " + str(len(signals)) + " signals found in '" + os.path.basename(path) +
              "', using #" + str(signal_index) + " '" + str(signals[signal_index].metadata.General.title) +
              "' " + str(signals[signal_index].data.shape))
    s = signals[signal_index]

    ax = s.axes_manager.signal_axes[0]
    unit_factor = UNIT_TO_NM.get(str(ax.units).lower(), 1.0)
    nm_per_px = float(ax.scale) * unit_factor

    om = s.original_metadata.as_dictionary()
    detector_name = om.get("BinaryResult", {}).get("Detector") or s.metadata.General.title
    is_darkfield = classify_darkfield(detector_name)

    gray = normalize_to_uint8(s.data)

    info = {
        "detector": detector_name,
        "raw_dtype": str(s.data.dtype),
        "raw_min": float(s.data.min()),
        "raw_max": float(s.data.max()),
        "raw": s.data.astype(np.float32),  # un-normalized intensities, used by the overlap filter
    }
    return gray, nm_per_px, is_darkfield, info

## 2. Segmentation (Otsu + watershed)

In [ ]:
def segment_and_split_particles(gray_img, min_peak_distance=15, bright_particles=False, blur_sigma=1.1):
    """
    Segments particles on a uniform background and splits touching
    particles using watershed based on the distance transform.

    Parameters
    ----------
    gray_img : np.ndarray (2D, uint8)
    min_peak_distance : int
        Minimum distance (in pixels) between local maxima of the distance
        transform, used as markers for the watershed. Should be chosen
        based on the approximate particle radius (smaller if particles are
        small/closely packed).
    bright_particles : bool
        False for bright-field-style images where particles are darker
        than the background. True for dark-field/HAADF STEM images, where
        particles are the brightest pixels (Z-contrast).
    blur_sigma : float
        Gaussian smoothing (in pixels) applied before thresholding. Values
        up to 1.1 keep the original light smoothing; larger values (used at
        high magnification, where particles are hundreds of pixels wide and
        the noisy edges would otherwise give very low circularity) also
        enlarge the morphological clean-up kernel.

    Returns
    -------
    labels_ws : np.ndarray (2D, int) - watershed labels (0 = background)
    mask : np.ndarray (2D, bool) - binary mask of the particles
    """
    if blur_sigma <= 1.1:
        blur = cv2.GaussianBlur(gray_img, (5, 5), 0)
        kernel = np.ones((3, 3), np.uint8)
    else:
        blur = cv2.GaussianBlur(gray_img, (0, 0), sigmaX=blur_sigma)
        k = int(2 * round(blur_sigma) + 1)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))

    thresh_type = cv2.THRESH_BINARY if bright_particles else cv2.THRESH_BINARY_INV
    _, th = cv2.threshold(blur, 0, 255, thresh_type + cv2.THRESH_OTSU)

    th_clean = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel, iterations=1)
    th_clean = cv2.morphologyEx(th_clean, cv2.MORPH_CLOSE, kernel, iterations=1)
    mask = th_clean > 0

    dist = ndi.distance_transform_edt(mask)
    coords = peak_local_max(dist, min_distance=int(max(1, min_peak_distance)), labels=mask,
                             exclude_border=False)
    markers_mask = np.zeros(dist.shape, dtype=bool)
    markers_mask[tuple(coords.T)] = True
    markers, _ = ndi.label(markers_mask)

    labels_ws = watershed(-dist, markers, mask=mask)
    return labels_ws, mask

## 3. Particle measurement and overlap filter

`measure_particles()` computes size, circularity and mean intensity of every watershed region and drops regions that are too small or not circular. `split_overlapping()` then separates overlapped particles using their intensity (dark-field images only): the HAADF signal adds up where particles overlap, so these are brighter than isolated particles.

In [ ]:
MIN_DIAM_PX_FLOOR = 6  # never accept regions smaller than this, however small min_diam_nm is (pure noise)


def measure_particles(labels_ws, nm_per_px, min_diam_px=25, min_circularity=0.5, intensity_img=None):
    """
    Computes area, equivalent diameter, circularity and (if intensity_img is
    given) mean intensity for each watershed region, and filters out regions
    that are too small or not circular enough (typically noise fragments or
    edge artifacts).

    Returns
    -------
    list of dict, one per accepted particle, with keys:
        label, area_px, eq_diam_px, eq_diam_nm, circularity, solidity,
        mean_intensity (None without intensity_img), centroid, bbox
    """
    props = regionprops(labels_ws, intensity_image=intensity_img)
    particles = []
    for p in props:
        area = p.area
        perim = p.perimeter if p.perimeter > 0 else 1
        circularity = 4 * np.pi * area / (perim ** 2)
        eq_diam_px = 2 * np.sqrt(area / np.pi)

        if eq_diam_px < min_diam_px:
            continue
        if circularity < min_circularity:
            continue

        particles.append({
            "label": p.label,
            "area_px": area,
            "eq_diam_px": eq_diam_px,
            "eq_diam_nm": eq_diam_px * nm_per_px,
            "circularity": circularity,
            "solidity": p.solidity,
            "mean_intensity": float(p.mean_intensity) if intensity_img is not None else None,
            "centroid": p.centroid,  # (row, col)
            "bbox": p.bbox,  # (min_row, min_col, max_row, max_col)
        })
    return particles


def split_overlapping(particles, max_intensity_ratio):
    """
    Splits particles into (kept, excluded) by mean intensity. In dark-field
    (HAADF) images the signal adds up where particles overlap, so overlapped
    particles are brighter than isolated ones. A particle is excluded when its
    mean intensity is above max_intensity_ratio times the median particle
    intensity of the image (e.g. 1.3 = more than 30% brighter than typical).
    """
    intensities = np.array([p["mean_intensity"] for p in particles])
    limit = max_intensity_ratio * np.median(intensities)
    kept = [p for p, v in zip(particles, intensities) if v <= limit]
    excluded = [p for p, v in zip(particles, intensities) if v > limit]
    return kept, excluded

## 4. Polydispersity statistics

In [ ]:
def polydispersity_stats(diam_nm):
    """
    Computes polydispersity statistics from a non-empty array of
    diameters (in nm). Only ever called with at least one particle --
    analyze_emd_image() handles the zero-particle case separately.

    PDI here is defined as (sigma/mean)^2, the same convention typically
    used in DLS. It is a dimensionless metric: lower values indicate a
    more monodisperse sample (e.g. PDI < 0.05 - 0.1 in TEM).
    """
    diam_nm = np.asarray(diam_nm)
    mean_d = diam_nm.mean()
    std_d = diam_nm.std(ddof=1) if diam_nm.size > 1 else 0.0
    return {
        "n": len(diam_nm),
        "mean_nm": mean_d,
        "median_nm": float(np.median(diam_nm)),
        "std_nm": std_d,
        "cv_percent": std_d / mean_d * 100,
        "pdi": (std_d / mean_d) ** 2,
        "min_nm": float(diam_nm.min()),
        "max_nm": float(diam_nm.max()),
    }

## 5. Visualization

In [ ]:
def draw_overlay(color_img, particles, out_path=None, excluded=None, border=None, oversized=None):
    """Kept: red circle + green centre. Excluded as overlapping: orange. Cut by the image border: cyan. Oversized: magenta."""
    overlay = color_img.copy()
    for p in (oversized or []):
        cy, cx = p["centroid"]
        r = p["eq_diam_px"] / 2
        cv2.circle(overlay, (int(round(cx)), int(round(cy))), int(round(r)),
                   (255, 0, 255), 2)
    for p in (border or []):
        cy, cx = p["centroid"]
        r = p["eq_diam_px"] / 2
        cv2.circle(overlay, (int(round(cx)), int(round(cy))), int(round(r)),
                   (255, 255, 0), 2)
    for p in (excluded or []):
        cy, cx = p["centroid"]
        r = p["eq_diam_px"] / 2
        cv2.circle(overlay, (int(round(cx)), int(round(cy))), int(round(r)),
                   (0, 165, 255), 2)
    for p in particles:
        cy, cx = p["centroid"]
        r = p["eq_diam_px"] / 2
        cv2.circle(overlay, (int(round(cx)), int(round(cy))), int(round(r)),
                   (0, 0, 255), 2)
        cv2.circle(overlay, (int(round(cx)), int(round(cy))), 1, (0, 255, 0), 2)
    if out_path:
        cv2.imwrite(out_path, overlay)
    return overlay


def show_overlay(overlay_bgr, n_particles=None, n_excluded=0, n_border=0, n_oversized=0):
    """Displays a BGR (OpenCV) overlay image inline using matplotlib."""
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB))
    title = "Detected particles"
    if n_particles is not None:
        title += f" (N={n_particles})"
    notes = []
    if n_excluded:
        notes.append(f"{n_excluded} overlapping (orange)")
    if n_border:
        notes.append(f"{n_border} cut by border (cyan)")
    if n_oversized:
        notes.append(f"{n_oversized} oversized (magenta)")
    if notes:
        title += "\nexcluded: " + ", ".join(notes)
    plt.title(title)
    plt.axis("off")
    plt.show()


def plot_histogram(diam_nm, stats, out_path=None, material="nanoparticles", show=True):
    plt.figure(figsize=(7, 5))
    plt.hist(diam_nm, bins=30, color="#4C72B0", edgecolor="white")
    plt.axvline(stats["mean_nm"], color="crimson", linestyle="--", linewidth=1.5,
                label=f'Mean = {stats["mean_nm"]:.2f} nm')
    plt.xlabel("Particle diameter (nm)")
    plt.ylabel("Count")
    plt.title(
        f'Size distribution {material} (N={stats["n"]})\n'
        f'Std dev = {stats["std_nm"]:.2f} nm, CV = {stats["cv_percent"]:.1f}%, '
        f'PDI = {stats["pdi"]:.4f}'
    )
    plt.legend()
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=150)
    if show:
        plt.show()
    else:
        plt.close()

## Full pipeline (single file)

`analyze_emd_image()` is purely computational (loads, segments, measures, computes stats) and never crashes on zero detected particles -- `stats`/`overlay` are simply `None` in that case. `show_results()` and `save_results()` handle displaying/saving, including showing the segmentation mask when nothing was detected, so you can see why.

In [ ]:
def touches_border(p, shape):
    """True if the particle's bounding box touches the image edge (the particle is cut by the frame)."""
    min_row, min_col, max_row, max_col = p["bbox"]
    return min_row == 0 or min_col == 0 or max_row >= shape[0] or max_col >= shape[1]


def analyze_emd_image(path, min_diam_nm=5, min_circularity=0.5, min_peak_distance_nm=5,
                       smooth_nm=0.3, exclude_border=True, max_intensity_ratio=1.3, max_diameter_ratio=2.5,
                       bright_particles_override=None, emd_signal_index=None):
    """
    Runs the full pipeline on a single .emd file (load + metadata
    calibration, segmentation, measurement, border/overlap filters,
    statistics) and returns a dict: path, nm_per_px, is_darkfield, detector,
    gray, mask, particles (kept), excluded (overlapping, dark-field only),
    border (cut by the image edge), oversized (diameter above
    max_diameter_ratio times the mean diameter), stats, overlay.
    If no particles are detected, stats and overlay are None -- use
    show_results() to see the segmentation mask and debug why (wrong
    bright/dark-field guess, or min_diam_nm/min_circularity too strict)
    instead of crashing.

    All the lengths (min_diam_nm, min_peak_distance_nm, smooth_nm) are
    converted to pixels with the file's own calibration, so the same values
    work at any magnification.
    """
    gray, nm_per_px, is_darkfield, info = load_emd_image(path, signal_index=emd_signal_index)
    intensity = info.pop("raw")
    if bright_particles_override is not None:
        is_darkfield = bright_particles_override

    blur_sigma = max(smooth_nm / nm_per_px, 1.1)
    min_peak_distance = max(min_peak_distance_nm / nm_per_px, 3)
    labels_ws, mask = segment_and_split_particles(
        gray, min_peak_distance=min_peak_distance, bright_particles=is_darkfield,
        blur_sigma=blur_sigma,
    )
    min_diam_px = max(min_diam_nm / nm_per_px, MIN_DIAM_PX_FLOOR)
    particles = measure_particles(
        labels_ws, nm_per_px=nm_per_px,
        min_diam_px=min_diam_px, min_circularity=min_circularity,
        intensity_img=intensity,
    )

    border = []
    if exclude_border and particles:
        border = [p for p in particles if touches_border(p, gray.shape)]
        particles = [p for p in particles if not touches_border(p, gray.shape)]

    excluded = []
    if max_intensity_ratio is not None and is_darkfield and particles:
        particles, excluded = split_overlapping(particles, max_intensity_ratio)

    oversized = []
    if max_diameter_ratio is not None and particles:
        limit = max_diameter_ratio * np.mean([q["eq_diam_nm"] for q in particles])
        oversized = [q for q in particles if q["eq_diam_nm"] > limit]
        particles = [q for q in particles if q["eq_diam_nm"] <= limit]

    stats = None
    overlay = None
    if particles:
        diam_nm = np.array([p["eq_diam_nm"] for p in particles])
        stats = polydispersity_stats(diam_nm)
    if particles or excluded or border or oversized:
        color_img = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        overlay = draw_overlay(color_img, particles, excluded=excluded, border=border, oversized=oversized)

    return {
        "path": path,
        "nm_per_px": nm_per_px,
        "is_darkfield": is_darkfield,
        "detector": info["detector"],
        "min_diam_px_used": min_diam_px,
        "gray": gray,
        "mask": mask,
        "particles": particles,
        "excluded": excluded,
        "border": border,
        "oversized": oversized,
        "stats": stats,
        "overlay": overlay,
    }


def show_results(result, material="nanoparticles"):
    """
    Displays the metadata summary and, if particles were found, the
    overlay + size histogram. If none were found, shows the segmentation
    mask instead so you can see why.
    """
    print("File: " + result["path"])
    print("Detector: " + str(result["detector"]) + " (dark-field: " + str(result["is_darkfield"]) + ")")
    print("Calibration: " + format(result["nm_per_px"], ".4f") + " nm/px" +
          " | min diameter filter: " + format(result["min_diam_px_used"], ".1f") + " px")
    print("Particles kept: " + str(len(result["particles"])) +
          " | excluded as overlapping: " + str(len(result["excluded"])) +
          " | cut by image border: " + str(len(result["border"])) +
          " | oversized: " + str(len(result["oversized"])))

    if not result["particles"]:
        print("No particles detected -- showing the segmentation mask for debugging. "
              "Try bright_particles_override, or relax min_diam_nm / min_circularity.")
        plt.figure(figsize=(8, 8))
        plt.imshow(result["mask"], cmap="gray")
        plt.title("Segmentation mask (no particles passed the filters)")
        plt.axis("off")
        plt.show()
        return

    show_overlay(result["overlay"], n_particles=result["stats"]["n"],
                 n_excluded=len(result["excluded"]), n_border=len(result["border"]),
                 n_oversized=len(result["oversized"]))
    diam_nm = np.array([p["eq_diam_nm"] for p in result["particles"]])
    plot_histogram(diam_nm, result["stats"], material=material)


def save_results(result, outdir, material="nanoparticles"):
    """Saves overlay.png and histogram.png for one result into outdir (skipped if no particles were found)."""
    if not result["particles"]:
        print("Skipping save for '" + result["path"] + "': no particles detected.")
        return
    os.makedirs(outdir, exist_ok=True)
    cv2.imwrite(os.path.join(outdir, "overlay.png"), result["overlay"])
    diam_nm = np.array([p["eq_diam_nm"] for p in result["particles"]])
    plot_histogram(diam_nm, result["stats"], os.path.join(outdir, "histogram.png"),
                    material=material, show=False)

## Run: single file

Uses `file_path` from Configuration. Runs only when `mode = "file"`.

In [ ]:
from IPython.display import display

if mode != "file":
    print("Skipped: mode = '" + mode + "'. Set mode = 'file' in Configuration to run this section.")
else:
    result = analyze_emd_image(
        file_path,
        min_diam_nm=min_diam_nm, max_intensity_ratio=max_intensity_ratio, max_diameter_ratio=max_diameter_ratio, min_circularity=min_circularity, min_peak_distance_nm=min_peak_distance_nm,
        smooth_nm=smooth_nm, exclude_border=exclude_border, emd_signal_index=emd_signal_index,
        bright_particles_override=bright_particles_override,
    )

    show_results(result)

    if outdir:
        save_results(result, outdir)

    display(result["stats"])

## Batch processing (multiple `.emd` files)

Uses `folder_path` from Configuration. Every `.emd` file directly inside that folder is analyzed; files with 0 detected particles are kept in the results (not skipped) so you can inspect them, files that error out unexpectedly (e.g. unreadable metadata) are skipped with a printed warning. Runs only when `mode = "folder"`.

In [ ]:
def analyze_emd_folder(folder_path, min_diam_nm=5, min_circularity=0.5, min_peak_distance_nm=5,
                        smooth_nm=0.3, exclude_border=True, max_intensity_ratio=1.3, max_diameter_ratio=2.5,
                        bright_particles_override=None, emd_signal_index=None):
    """
    Runs analyze_emd_image() on every .emd file directly inside
    folder_path (non-recursive) and returns a dict {file_name: result}.
    """
    emd_paths = sorted(glob.glob(os.path.join(folder_path, "*.emd")))
    if not emd_paths:
        raise FileNotFoundError("No .emd files found in: " + folder_path)

    results = {}
    for path in emd_paths:
        name = os.path.splitext(os.path.basename(path))[0]
        try:
            res = analyze_emd_image(
                path, min_diam_nm=min_diam_nm, min_circularity=min_circularity,
                min_peak_distance_nm=min_peak_distance_nm, smooth_nm=smooth_nm,
                exclude_border=exclude_border, max_intensity_ratio=max_intensity_ratio,
                max_diameter_ratio=max_diameter_ratio,
                bright_particles_override=bright_particles_override,
                emd_signal_index=emd_signal_index,
            )
        except Exception as exc:
            print("[SKIPPED] " + name + ": " + str(exc))
            continue

        results[name] = res
        if res["stats"] is None:
            print(name + ": 0 particles detected (detector: " + str(res["detector"]) +
                  ", dark-field: " + str(res["is_darkfield"]) + ")")
        else:
            s = res["stats"]
            print(name + ": N=" + str(s["n"]) + " (excluded: " + str(len(res["excluded"])) + " overlapping, " +
                  str(len(res["border"])) + " cut by border, " +
                  str(len(res["oversized"])) + " oversized)" +
                  ", mean=" + format(s["mean_nm"], ".2f") +
                  " nm, CV=" + format(s["cv_percent"], ".1f") + "%, PDI=" + format(s["pdi"], ".4f"))

    return results


def summarize_folder_results(results):
    """Builds a one-row-per-file summary table (rows with 0 particles show n=0 and NaN elsewhere)."""
    rows = []
    for name, res in results.items():
        row = {"image": name, "detector": res["detector"], "is_darkfield": res["is_darkfield"],
               "n_excluded_overlap": len(res["excluded"]), "n_excluded_border": len(res["border"]),
               "n_excluded_oversized": len(res["oversized"])}
        if res["stats"] is not None:
            row.update(res["stats"])
        else:
            row["n"] = 0
        rows.append(row)
    return pd.DataFrame(rows).set_index("image").sort_index()

### Run batch analysis

In [ ]:
from IPython.display import display

if mode != "folder":
    print("Skipped: mode = '" + mode + "'. Set mode = 'folder' in Configuration to run this section.")
else:
    folder_results = analyze_emd_folder(
        folder_path,
        min_diam_nm=min_diam_nm, max_intensity_ratio=max_intensity_ratio, max_diameter_ratio=max_diameter_ratio, min_circularity=min_circularity, min_peak_distance_nm=min_peak_distance_nm,
        smooth_nm=smooth_nm, exclude_border=exclude_border, emd_signal_index=emd_signal_index,
        bright_particles_override=bright_particles_override,
    )

    for name, res in folder_results.items():
        show_results(res)
        if outdir:
            save_results(res, os.path.join(outdir, name))

    summary = summarize_folder_results(folder_results)
    display(summary)

### Combined size distribution (all files pooled)

In [ ]:
from IPython.display import display

if mode != "folder":
    print("Skipped: mode = '" + mode + "'. Set mode = 'folder' in Configuration to run this section.")
else:
    particle_groups = [
        np.array([p["eq_diam_nm"] for p in res["particles"]])
        for res in folder_results.values() if res["particles"]
    ]
    if particle_groups:
        all_diam_nm = np.concatenate(particle_groups)
        combined_stats = polydispersity_stats(all_diam_nm)
        plot_histogram(
            all_diam_nm, combined_stats,
            out_path=os.path.join(outdir, "combined_histogram.png") if outdir else None,
        )
        display(combined_stats)
    else:
        print("No particles detected in any file -- nothing to combine.")